In [ ]:
# Architecture Overview

# User Question (English or Roman Urdu)
#         ↓
#    Language Detection
#         ↓
#    Query Embedding  ←── FAISS Vector Store ←── Knowledge Base Docs
#         ↓
#    Top-K Retrieval
#         ↓
#   Groq LLM (Llama 3.3 70B)  ←── Context + User Query
#         ↓
#    Response (same language as input)

In [1]:
# Step 1 — Install Dependencies

!pip install groq faiss-cpu sentence-transformers langchain langchain-community \
             langdetect gradio -q
print("All RAG dependencies installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 9.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
All RAG dependencies installed!


In [1]:
# Step 2 — Build Your Knowledge Base
# knowledge_base.py  (or paste directly in a cell)

RICE_KNOWLEDGE = """
=== RICE DISEASES IN PAKISTAN ===

--- Bacterial Leaf Blight (BLB) ---
Bacterial Leaf Blight (BLB) is caused by Xanthomonas oryzae pv. oryzae.
It is one of the most destructive rice diseases in Punjab and Sindh, Pakistan.
Symptoms: Water-soaked to yellowish stripes on leaf margins, leaves dry out from tip.
Conditions: Hot humid weather above 30°C, waterlogged fields, excess nitrogen.
Spread: Through infected water, rain splash, farm tools.
Immediate action: Remove infected leaves, stop nitrogen fertilizer, improve drainage.
Treatment: Spray Copper Oxychloride 50% WP at 2.5g per liter every 10-14 days.
Also use Streptomycin Sulphate 90% SP at 0.5g per liter, 2 sprays 7 days apart.
Kasugamycin 3% SL at 2mL per liter, spray every 10 days maximum 3 times.
Fertilizer: Reduce nitrogen (Urea), use Potassium Chloride (KCl) 60kg per hectare.
Use Diammonium Phosphate (DAP) 50kg per hectare at basal application only.
Prevention: Use resistant varieties, clean seeds, proper field drainage, avoid excess urea.

--- Brown Spot ---
Brown Spot is caused by Helminthosporium oryzae (fungal disease).
Common in nutrient-deficient soils, especially in Sindh and southern Punjab.
Symptoms: Oval to circular brown spots on leaves with yellow halo, spots on grains too.
Conditions: Low soil fertility, silicon deficiency, drought stress, high humidity.
Treatment: Spray Mancozeb 75% WP at 2g per liter water, repeat after 14 days.
Also use Propiconazole 25% EC at 1mL per liter, 2-3 applications.
Tricyclazole 75% WP at 0.6g per liter is also effective.
Fertilizer: Apply balanced NPK fertilizer. Potassium silicate helps resistance.
Use Silicon-based fertilizers to strengthen cell walls.
Prevention: Use certified disease-free seeds, balanced fertilization, avoid water stress.

--- Leaf Smut ---
Leaf Smut is caused by Entyloma oryzae (fungal disease).
Less severe than BLB but causes quality loss. Found in humid regions of Pakistan.
Symptoms: Small angular black spots on leaves, powdery spore masses.
Treatment: Spray Carbendazim 50% WP at 1g per liter, or Mancozeb 75% WP at 2g per liter.
Prevention: Crop rotation, remove plant debris, use clean seeds treated with fungicide.

=== COMMON PESTICIDES FOR RICE IN PAKISTAN ===

--- Insecticides ---
Chlorpyrifos 40% EC: Controls stem borers, leaf folders. Use 1.5L per hectare.
Imidacloprid 200 SL: Controls brown planthopper (BPH). Use 200mL per hectare.
Lambda-cyhalothrin 2.5% EC: Controls leaf folders and BPH. Use 300mL per hectare.
Fipronil 5% SC: Controls stem borers. Use 1L per hectare.
Cartap Hydrochloride 50% SP: Controls stem borers, leaf folders. Use 1kg per hectare.

--- Common Rice Pests in Pakistan ---
Stem Borer (Scirpophaga): Most damaging pest in Punjab. Causes dead heart and white ear.
Treatment: Chlorpyrifos spray + Carbofuran 3G granules 10kg per hectare at tillering.
Brown Planthopper (BPH): Causes hopper burn. Use Imidacloprid or Buprofezin.
Leaf Folder: Rolls leaves into tubes. Use Lambda-cyhalothrin or Chlorpyrifos.
Rice Hispa: Scrapes leaf surface. Use Malathion 57% EC at 1L per hectare.

=== FERTILIZERS FOR RICE IN PAKISTAN ===

--- General NPK Schedule ---
Basal (Before transplanting): DAP 100kg/ha + SOP or MOP 50kg/ha
1st Top Dress (21-25 days after transplant): Urea 65kg/ha
2nd Top Dress (45 days after transplant): Urea 65kg/ha
Total Nitrogen: 120kg N per hectare for high-yield varieties.

--- Recommended Varieties in Pakistan ---
Basmati 515: Popular in Punjab, aromatic, disease tolerant.
Super Basmati: Export quality, but susceptible to BLB.
IRRI-6: High yield, tolerant to many diseases, common in Sindh.
KSK-282: Resistant to BLB, suitable for Punjab.
PK-386: Suitable for Sindh conditions.

--- Soil Preparation ---
Plow 2-3 times to 20cm depth. Level field for uniform irrigation.
Apply Farm Yard Manure (FYM) 5-10 tons per hectare before final plowing.
Soil pH should be 5.5 to 7.0 for best rice growth.

=== IRRIGATION MANAGEMENT ===
Maintain 5-7cm water depth during vegetative stage.
Use Alternate Wetting and Drying (AWD) to save water and reduce BLB risk.
Stop irrigation 10 days before harvest.
Avoid waterlogging as it promotes bacterial diseases.

=== PLANTING SEASON IN PAKISTAN ===
Nursery sowing: Late May to mid-June (Punjab), May-June (Sindh).
Transplanting: Late June to mid-July when seedlings are 25-30 days old.
Harvesting: October to November when 80% grains are mature.

=== ROMAN URDU COMMON TERMS ===
Chawal ki fasal = Rice crop
Bimari = Disease
Keeray = Insects/Pests
Khad = Fertilizer
Spray = Pesticide spray
Paani = Water
Zameen = Soil/Land
Mosam = Weather
Paidawar = Yield/Production
Bijli wali machine = Electric pump
Khal = Canal
Boring = Tube well
"""

In [2]:
# Step 3 — Build the RAG Pipeline

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
import numpy as np

# ── 1. Split knowledge into chunks ──
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,        # characters per chunk
    chunk_overlap=100,     # overlap so context isn't cut off
    separators=["\n\n", "\n", "---", "===", ". "]
)
chunks = text_splitter.create_documents([RICE_KNOWLEDGE])
print(f"Knowledge split into {len(chunks)} chunks")

# ── 2. Create embeddings (free, runs locally) ──
# This downloads ~90MB model once, then caches it
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}   # embeddings don't need GPU
)
print("Embedding model loaded")

# ── 3. Build FAISS vector store ──
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})  # top 4 chunks
print("Vector store ready!")

# ── Optional: Save to Drive so you don't rebuild every session ──
vectorstore.save_local("/content/drive/MyDrive/rice_vectorstore")
# To reload later: vectorstore = FAISS.load_local("/content/drive/MyDrive/rice_vectorstore", embeddings)


Knowledge split into 14 chunks


/tmp/ipykernel_43887/829462417.py:19: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Vector store ready!


In [3]:
# Step 4 — Language Detection + LLM Response

from groq import Groq
from langdetect import detect

client = Groq(api_key="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")

def detect_language(text):
    roman_urdu_words = ['kya', 'hai', 'hain', 'nahi', 'karo', 'mera', 'meri',
                        'chawal', 'fasal', 'bimari', 'khad', 'keeray', 'acha',
                        'batao', 'kaise', 'kyun', 'kab', 'kahan', 'kitna']
    text_lower = text.lower()
    roman_hits = sum(1 for w in roman_urdu_words if w in text_lower)
    if roman_hits >= 2:
        return 'roman_urdu'
    try:
        lang = detect(text)
        if lang == 'ur':
            return 'urdu'
    except:
        pass
    return 'english'


def get_rag_response(user_question, chat_history=[]):
    # 1. Detect language
    lang = detect_language(user_question)

    # 2. Retrieve context — FIXED: use .invoke() instead of deprecated method
    docs = retriever.invoke(user_question)
    context = "\n\n".join([doc.page_content for doc in docs])

    # 3. Language instruction
    if lang == 'roman_urdu':
        lang_instruction = (
            "The farmer is asking in Roman Urdu. "
            "Reply in simple Roman Urdu. Example: 'Aap ki fasal mein BLB bimari hai. Copper spray karein.'"
        )
    elif lang == 'urdu':
        lang_instruction = "Reply in Urdu script."
    else:
        lang_instruction = "Reply in clear, simple English suitable for farmers."

    # 4. System prompt
    system_prompt = f"""You are an expert agricultural assistant for Pakistani rice farmers.
You specialize in rice crop diseases, pesticides, fertilizers, irrigation, and farming practices
specifically for Pakistan's Punjab and Sindh regions.

{lang_instruction}

Use ONLY the context provided below to answer. If the answer is not in the context, say you don't
have that specific information but give general safe advice.

Keep answers practical and actionable. Mention specific product names, dosages, and timing when available.
Always remind farmers to consult their local Agriculture Extension Officer for serious issues.

CONTEXT FROM KNOWLEDGE BASE:
{context}
"""

    # 5. Build messages — FIXED: handle both old tuple format and new dict format
    messages = [{"role": "system", "content": system_prompt}]

    for entry in chat_history[-6:]:
        if isinstance(entry, dict):
            # New Gradio format: {"role": "user"/"assistant", "content": "..."}
            messages.append({"role": entry["role"], "content": entry["content"]})
        elif isinstance(entry, (list, tuple)) and len(entry) == 2:
            # Old tuple format: [user_msg, bot_msg]
            if entry[0]:
                messages.append({"role": "user", "content": entry[0]})
            if entry[1]:
                messages.append({"role": "assistant", "content": entry[1]})

    messages.append({"role": "user", "content": user_question})

    # 6. Call Groq
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
        temperature=0.3,
        max_tokens=800
    )

    return response.choices[0].message.content


# Quick test BEFORE launching UI — run this to confirm it works
print("Testing RAG pipeline...")
test = get_rag_response("What is bacterial leaf blight and how to treat it?")
print("Test passed! Response preview:")
print(test[:300], "...")

Testing RAG pipeline...
Test passed! Response preview:
Bacterial Leaf Blight (BLB) is a very destructive rice disease in Punjab and Sindh, Pakistan. It is caused by a bacteria called Xanthomonas oryzae pv. oryzae. 

You can identify BLB by looking for water-soaked to yellowish stripes on the edges of the leaves. As the disease gets worse, the leaves wil ...


In [5]:
# Step 5 — Gradio Chat UI

import gradio as gr

def chat_fn(message, history):
    return get_rag_response(message, history)

# FIXED: ChatInterface standalone (not nested in Blocks)
demo = gr.ChatInterface(
    fn=chat_fn,
    type="messages",          # ← FIXES the tuple deprecation warning
    title="🌾 Pakistan Rice Farming Assistant",
    description=(
        "### Chawal Ki Fasal Ka AI Maahir | آپ کا ذہین زرعی معاون\n"
        "Ask in **English** or **Roman Urdu** — e.g. *'BLB bimari ka ilaj kya hai?'*\n\n"
        "⚠️ *Always consult your local Agriculture Extension Officer before applying chemicals.*"
    ),
    examples=[
        "What is bacterial leaf blight and how to treat it?",
        "Chawal mein brown spot bimari ka ilaj kya hai?",
        "Rice mein kaunsi khad dalni chahiye?",
        "Stem borer keeray ko kaise khatam karein?",
        "Best rice variety for Punjab Pakistan?",
        "Chawal ki fasal mein paani kitna dena chahiye?",
    ],
    theme=gr.themes.Soft(),
)

demo.launch(share=True, debug=True)   # ← debug=True so you SEE errors instead of just "Error"


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3570173f8c33dad53f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://3570173f8c33dad53f.gradio.live
